# WaveFlux-CeNN × SmolLM2-360M-Instruct: replace first attention layer

**Experiment, not a validated replacement.** The `WaveFluxCeNN` submodule is **pure cellular** (no attention, Q/K/V, LSTM, DeltaNet or mixing with native output). We replace **only `model.layers[0].self_attn`**; the *rest of the 360M Transformer* is deliberately retained as an evaluation harness. The **complete model is therefore not yet attention-free**. Native attention is computed only for frozen teacher targets / paired baseline, **never in the replacement inference path**.

The train/dev/test partitions are disjoint WikiText-2 splits. Only the **development** set selects the checkpoint; test is read once afterwards. See [evaluation protocol](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/docs/WAVEFLUX_SMOL360_LAYER0.md).

In [ ]:
# @title 1. Runtime and dependency setup
import os, subprocess, sys
import torch
print('PyTorch:',torch.__version__)
print('CUDA:',torch.cuda.is_available(),torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → T4 GPU (or faster)'
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.6','datasets>=4.0,<5','huggingface_hub>=0.34,<2','safetensors','sentencepiece','pytest'],check=True)
if not os.path.isdir('/content/TinyCeNN-LM'):
    subprocess.run(['git','clone','--depth','1','https://github.com/vtavakkoli/TinyCeNN-LM.git','/content/TinyCeNN-LM'],check=True)
else:
    subprocess.run(['git','-C','/content/TinyCeNN-LM','pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','/content/TinyCeNN-LM','--no-deps'],check=True)
print('Repository revision:',subprocess.check_output(['git','-C','/content/TinyCeNN-LM','rev-parse','HEAD'],text=True).strip())

In [ ]:
# @title 2. Check mathematical and attention-slot contracts
import subprocess, sys
subprocess.run([sys.executable,'-m','pytest','-q',
 '/content/TinyCeNN-LM/tests/test_waveflux_cenn.py',
 '/content/TinyCeNN-LM/tests/test_waveflux_smol360_contract.py'],check=True)

In [ ]:
# @title 3. Research configuration
PROFILE = 'quick' # @param ['smoke','quick','research']
SEED = 2031 # @param {type:'integer'}
GROUPS = 24 # @param {type:'integer'}
CELL_WIDTH = 32 # @param {type:'integer'}
STEPS_OVERRIDE = 0 # @param {type:'integer'}
OUTPUT_DIR = '/content/waveflux_smol360_l0'
print('Training configuration',dict(profile=PROFILE,seed=SEED,groups=GROUPS,cell_width=CELL_WIDTH,steps_override=STEPS_OVERRIDE))

In [ ]:
# @title 4. Frozen-teacher IO distillation, validation selection, sealed test, CUDA timings
import os, subprocess, sys
command=[sys.executable,'/content/TinyCeNN-LM/scripts/benchmark_waveflux_smol360.py',
 '--profile',PROFILE,'--seed',str(SEED),'--groups',str(GROUPS),
 '--cell-width',str(CELL_WIDTH),'--output-dir',OUTPUT_DIR]
if STEPS_OVERRIDE>0: command += ['--steps',str(STEPS_OVERRIDE)]
env={**os.environ,'PYTHONUNBUFFERED':'1'}
subprocess.run(command,check=True,cwd='/content/TinyCeNN-LM',env=env)


In [ ]:
# @title 5. Review accuracy, NLL and measured GPU speed
from pathlib import Path
import json
result=json.loads((Path(OUTPUT_DIR)/'final_report.json').read_text())
for key in ('model','revision','replaced_layer','original_attention_params','replacement_params','best_step','dev_quality_pass','test_quality_pass','accepted'):
    print(f'{key}: {result[key]}')
print('DEV',json.dumps(result['dev'],indent=2))
print('TEST',json.dumps(result['test'],indent=2))
print('GPU latency',json.dumps(result['latency'],indent=2))
if not result['accepted']:
    print('NO ACCEPTANCE: keep original layer 0. This adapter is an EXPERIMENT, not a deployable model.')

In [ ]:
# @title 6. Export complete diagnostic ZIP (no model upload)
import shutil
from pathlib import Path
from google.colab import files
archive=shutil.make_archive(OUTPUT_DIR,'zip',root_dir=OUTPUT_DIR)
print('Diagnostics + experimental adapter:',archive)
files.download(archive)

## Interpretation

Pass/fail should reflect **whole-model next-token behavior and full-model prefill latency**, not isolated layer fit. No Hugging Face model publication or claims of standalone 360M conversion are made. The adapter does not support cached generation, masks/padding, or all-layer conversion. Use the research profile and further external holdout tasks only after the first controlled feasibility screen.